In [ ]:
# -*- coding: utf-8 -*-
"""
Ultimate Reviewer Rebuttal Package: 
1. Stacking Meta-Learner Weights
2. Expected Calibration Error (ECE)
3. E-Value for Causality (Albumin Paradox)
"""

import os
import pandas as pd
import numpy as np
import joblib
import warnings

warnings.filterwarnings('ignore')

print("="*85)
print("🛡️ RUNNING ULTIMATE REVIEWER REBUTTAL PACKAGE 🛡️")
print("="*85)

# ==========================================
# 基础配置与数据加载
# ==========================================
BASE_DIR = os.getcwd()
DATA_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'test_imputed_random_forest.csv')
STACKING_MODEL_FILE = os.path.join(BASE_DIR, 'Final_Internal_Validation_Optimized', 'model_stacking.pkl')
TARGET_COL = 'PostopAKI'

# 尝试加载数据和模型
try:
    df_test = pd.read_csv(DATA_FILE)
    if 'Unnamed: 0' in df_test.columns: df_test.drop(columns=['Unnamed: 0'], inplace=True)
    if 'ID' in df_test.columns: df_test.drop(columns=['ID'], inplace=True)
    
    y_true = df_test[TARGET_COL].astype(int).values
    
    model_stacking, expected_cols = joblib.load(STACKING_MODEL_FILE)
    X_test_stark = df_test[expected_cols].fillna(0)
    
    # 获取 STARK 预测概率
    stark_probs = model_stacking.predict_proba(X_test_stark)[:, 1]
    print("✅ 数据与 STARK 模型加载成功，预测完成。")
except Exception as e:
    print(f"❌ 加载失败，请检查路径。报错信息: {e}")
    exit()

print("\n" + "-"*85)
# ==========================================
# 🎯 分析 1：Stacking 元学习器权重拆解 (Table S12)
# ==========================================
print("▶️ ANALYSIS 1: Extracting Stacking Meta-Learner Weights")

try:
    meta_learner = model_stacking.final_estimator_
    
    # 动态获取基础模型的名称 (防呆设计，避免名称对不上)
    if hasattr(model_stacking, 'named_estimators_'):
        base_model_names = list(model_stacking.named_estimators_.keys())
    else:
        base_model_names = ["LR", "DT", "RF", "KNN", "SVM", "NB", "XGBoost", "SGBT", "NNET"] 
    
    weights = None
    if hasattr(meta_learner, 'coef_'):
        weights = meta_learner.coef_[0]
    elif hasattr(meta_learner, 'feature_importances_'):
        weights = meta_learner.feature_importances_
    
    if weights is not None:
        weight_df = pd.DataFrame({
            'Base Model': base_model_names,
            'Meta-Learner Weight (Beta/Importance)': weights
        }).sort_values(by='Meta-Learner Weight (Beta/Importance)', ascending=False)
        
        print("\n📊 Table S12: Contribution of Base Learners in STARK Architecture")
        print(weight_df.to_markdown(index=False))
        
        weight_df.to_excel("Table_S12_Stacking_Weights.xlsx", index=False)
        print("\n✅ 权重表已保存为: Table_S12_Stacking_Weights.xlsx")
    else:
        print("⚠️ 当前元学习器不支持提取 coef_ 或 feature_importances_。")
except Exception as e:
    print(f"❌ 权重提取失败: {e}")

print("\n" + "-"*85)
# ==========================================
# 🎯 分析 2：Expected Calibration Error (ECE)
# ==========================================
print("▶️ ANALYSIS 2: Expected Calibration Error (ECE) Calculation")

def calculate_ece(y_true, y_prob, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]
        in_bin = (y_prob > bin_lower) & (y_prob <= bin_upper)
        prop_in_bin = np.mean(in_bin)
        
        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(y_true[in_bin])
            avg_confidence_in_bin = np.mean(y_prob[in_bin])
            ece += prop_in_bin * np.abs(avg_confidence_in_bin - accuracy_in_bin)
    return ece

stark_ece = calculate_ece(y_true, stark_probs, n_bins=10)
print(f"\n📏 STARK Expected Calibration Error (ECE): {stark_ece:.4f}")
if stark_ece < 0.05:
    print("   💡 点评: ECE < 0.05，极度优秀！概率校准极其精准，可直接指导临床决策。")
else:
    print("   💡 点评: 表现良好，请将此数值补充至 Table 7 的 Brier Score 旁边。")

print("\n" + "-"*85)
# ==========================================
# 🎯 分析 3：E-Value 因果混杂防御 (白蛋白悖论)
# ==========================================
print("▶️ ANALYSIS 3: E-Value Calculation for Unmeasured Confounding")

# 🌟 极其重要：请在这里填入你多因素回归算出来的白蛋白 OR 值和上下限 🌟
# 假设多因素回归中，白蛋白超量的 OR 为 2.45，95% CI 为 1.80 ~ 3.35
albumin_OR = 2.45        # 替换为你的真实 OR
albumin_CI_lower = 1.80  # 替换为你的真实 CI 下限
albumin_CI_upper = 3.35  # 替换为你的真实 CI 上限

def calculate_e_value(or_estimate):
    """
    计算基于 VanderWeele (2017) 的 E-Value 公式
    如果 OR < 1，则转换为 1/OR 计算。
    """
    if or_estimate < 1:
        rr = 1.0 / or_estimate
    else:
        rr = or_estimate
    # 对于相对常见结局，通常使用近似: E = RR + sqrt(RR * (RR - 1))
    e_value = rr + np.sqrt(rr * (rr - 1))
    return e_value

try:
    e_val_estimate = calculate_e_value(albumin_OR)
    e_val_lower = calculate_e_value(albumin_CI_lower) if albumin_OR > 1 else calculate_e_value(albumin_CI_upper)
    
    print(f"\n💉 【输入参数】白蛋白 > 25g (OR = {albumin_OR:.2f}, 95% CI: {albumin_CI_lower:.2f}-{albumin_CI_upper:.2f})")
    print(f"🛡️  【计算结果】E-Value (Point Estimate) = {e_val_estimate:.2f}")
    print(f"🛡️  【计算结果】E-Value (Lower Bound)    = {e_val_lower:.2f}")
    
    print("\n📝 【向审稿人解释的话术 (可直接粘贴至论文/回复信)】:")
    print(f"\"To address concerns of unmeasured confounding regarding intraoperative albumin administration (>25g), "
          f"we computed the E-value. The point estimate E-value is {e_val_estimate:.2f} (lower bound: {e_val_lower:.2f}). "
          f"This indicates that an unmeasured confounder (e.g., hidden disease severity or unrecorded massive bleeding) "
          f"would need to be associated with BOTH albumin administration and PO-AKI by a risk ratio of at least {e_val_lower:.2f}-fold "
          f"each, above and beyond the measured covariates, to fully explain away the observed specific odds ratio ({albumin_OR:.2f}). "
          f"Given the comprehensive inclusion of intraoperative blood loss, fluid volume, and ASA grades in our multivariate model, "
          f"the existence of such a massive hidden confounder is highly improbable, reinforcing the robustness of our finding.\"")

except Exception as e:
    print(f"❌ E-Value 计算失败，请检查输入的 OR 值是否为有效数字: {e}")

print("\n" + "="*85)
print("🎉 ALL REBUTTAL ANALYSES COMPLETED SUCCESSFULLY 🎉")
print("="*85)

In [ ]:
# -*- coding: utf-8 -*-
"""
Ultimate Clinical & Fairness Audit Package for npj Digital Medicine
1. Fairness Audit (Equalized Odds) - [Table S16 + Figure S7 Bar Chart]
2. Global Permutation Importance - [Figure S6 Bar Chart]
3. Missingness Pattern Test - [Console Output]
4. Clinical Utility Simulation & DCA - [Table 8 + Figure 8 DCA Curve]
5. Error Audit / Failure Mode - [Excel Report]
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import warnings

warnings.filterwarnings('ignore')

print("="*85)
print("🛡️ RUNNING ULTIMATE CLINICAL & FAIRNESS AUDIT PACKAGE (npj Edition) 🛡️")
print("="*85)

# ==========================================
# 0. 基础配置、数据加载与输出文件夹准备
# ==========================================
BASE_DIR = os.getcwd()
DATA_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'test_imputed_random_forest.csv')
ORIGINAL_DATA_FILE = os.path.join(BASE_DIR, 'clean_clinical_data.csv')
STACKING_MODEL_FILE = os.path.join(BASE_DIR, 'Final_Internal_Validation_Optimized', 'model_stacking.pkl')

TARGET_COL = 'PostopAKI'
CLINICAL_THRESHOLD = 0.15 # 论文推荐的床旁预警红线

# 🌟 创建独立的输出文件夹 🌟
OUTPUT_DIR = os.path.join(BASE_DIR, "npj_Rebuttal_Outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 所有图表和数据将统一保存在独立文件夹中: {OUTPUT_DIR}\n")

try:
    df_test = pd.read_csv(DATA_FILE)
    df_orig = pd.read_csv(ORIGINAL_DATA_FILE) 
    
    # 利用 Unnamed: 0 恢复原始性别和年龄特征方便切分
    if 'Unnamed: 0' in df_test.columns:
        df_orig['original_index'] = df_orig.index
        df_test['original_index'] = df_test['Unnamed: 0']
        df_test = df_test.merge(df_orig[['original_index', 'Gender', 'Age']], on='original_index', how='left', suffixes=('', '_orig'))
        if 'Age_orig' in df_test.columns:
            df_test['Age'] = df_test['Age_orig']
        if 'Gender_orig' in df_test.columns:
            df_test['Original_Gender'] = df_test['Gender_orig']
    
    y_true = df_test[TARGET_COL].astype(int).values
    model_stacking, expected_cols = joblib.load(STACKING_MODEL_FILE)
    X_test_stark = df_test[expected_cols].fillna(0)
    
    # 计算 STARK 概率
    stark_probs = model_stacking.predict_proba(X_test_stark)[:, 1]
    df_test['STARK_Prob'] = stark_probs
    print("✅ 数据与 STARK 模型加载成功，预测完成。")
except Exception as e:
    print(f"❌ 加载失败，请检查路径。报错信息: {e}")
    exit()

# ==========================================
# 🎯 分析 1：算法公平性审计 (图表并茂版)
# ==========================================
print("\n▶️ ANALYSIS 1: Algorithmic Fairness Audit (Equalized Odds)")

def calc_fairness_metrics(y, probs, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    return fnr, fpr

fairness_results = []
subgroups = [
    ("Age >= 65", df_test[df_test['Age'] >= 65]),
    ("Age < 65",  df_test[df_test['Age'] < 65]),
]
if 'Original_Gender' in df_test.columns:
    subgroups.extend([
        ("Male",   df_test[df_test['Original_Gender'].isin(['Male', '男', 1, '1'])]),
        ("Female", df_test[df_test['Original_Gender'].isin(['Female', '女', 0, '0'])])
    ])

for group_name, sub_df in subgroups:
    if len(sub_df) == 0: continue
    y_sub = sub_df[TARGET_COL].astype(int).values
    probs_sub = sub_df['STARK_Prob'].values
    fnr, fpr = calc_fairness_metrics(y_sub, probs_sub, CLINICAL_THRESHOLD)
    fairness_results.append({
        'Subgroup': group_name, 'N': len(sub_df), 
        'FNR': fnr, 'FPR': fpr 
    })

df_fairness = pd.DataFrame(fairness_results)

# 1. 保存格式化表格
df_fairness_out = df_fairness.copy()
df_fairness_out['FNR (Miss Rate)'] = df_fairness_out['FNR'].apply(lambda x: f"{x:.3f}")
df_fairness_out['FPR (False Alarm)'] = df_fairness_out['FPR'].apply(lambda x: f"{x:.3f}")
excel_fairness_path = os.path.join(OUTPUT_DIR, "Table_S16_Fairness_Audit.xlsx")
df_fairness_out.drop(columns=['FNR', 'FPR']).to_excel(excel_fairness_path, index=False)
print(f"✅ Table S16 (公平性表格) 已生成。")

# 2. 绘制公平性对比图
fig, ax = plt.subplots(figsize=(9, 6), dpi=300)
x = np.arange(len(df_fairness))
width = 0.35

rects1 = ax.bar(x - width/2, df_fairness['FNR'], width, label='False Negative Rate (Miss)', color='#1f77b4', alpha=0.85)
rects2 = ax.bar(x + width/2, df_fairness['FPR'], width, label='False Positive Rate (Alarm)', color='#ff7f0e', alpha=0.85)

ax.set_ylabel('Error Rate', fontsize=12, fontweight='bold')
ax.set_title(f'Algorithmic Fairness Across Demographics (Threshold = {CLINICAL_THRESHOLD*100}%)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(df_fairness['Subgroup'], fontsize=11)
ax.legend(loc='upper left')
ax.grid(axis='y', linestyle=':', alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig_fairness_path = os.path.join(OUTPUT_DIR, "Figure_S7_Fairness_Audit.png")
plt.savefig(fig_fairness_path, dpi=300, bbox_inches='tight')
print(f"✅ Figure S7 (公平性视觉图) 已生成。")

# ==========================================
# 🎯 分析 2：全局特征置换重要性 (Permutation Importance)
# ==========================================
print("\n▶️ ANALYSIS 2: Global Permutation Importance (F1-score Drop)")

base_preds = (stark_probs >= 0.5).astype(int)
base_f1 = f1_score(y_true, base_preds, zero_division=0)

importances = []
print("   (Calculating permutations... this takes a few seconds)")
top_features = expected_cols[:30] if len(expected_cols) > 30 else expected_cols

for col in top_features:
    X_permuted = X_test_stark.copy()
    X_permuted[col] = np.random.permutation(X_permuted[col].values)
    perm_probs = model_stacking.predict_proba(X_permuted)[:, 1]
    perm_preds = (perm_probs >= 0.5).astype(int)
    perm_f1 = f1_score(y_true, perm_preds, zero_division=0)
    drop_val = base_f1 - perm_f1
    importances.append({'Feature': col, 'F1 Drop': drop_val})

df_imp = pd.DataFrame(importances).sort_values(by='F1 Drop', ascending=True).tail(15) 

fig, ax = plt.subplots(figsize=(9, 7), dpi=300)
ax.barh(df_imp['Feature'], df_imp['F1 Drop'], color='#d62728', alpha=0.85)
ax.set_xlabel('Decrease in F1-Score (Permutation Importance)', fontsize=12, fontweight='bold')
ax.set_title('Global Feature Importance via Target Permutation', fontsize=14, fontweight='bold', pad=15)
ax.grid(axis='x', linestyle=':', alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig_imp_path = os.path.join(OUTPUT_DIR, "Figure_S6_Permutation_Importance.png")
plt.savefig(fig_imp_path, dpi=300, bbox_inches='tight')
print(f"✅ Figure S6 (置换重要性条形图) 已生成。")

# ==========================================
# 🎯 分析 3：缺失值模式假信号排除测试 (Missingness Pattern Test)
# ==========================================
print("\n▶️ ANALYSIS 3: Missingness Pattern Fake Signal Test")
try:
    orig_features = [c for c in expected_cols if c in df_orig.columns]
    X_missing = df_orig[orig_features].isnull().astype(int)
    y_missing_target = df_orig[TARGET_COL].fillna(0).astype(int).values
    
    lr_missing = LogisticRegression(class_weight='balanced', max_iter=200)
    cv_auc = cross_val_score(lr_missing, X_missing, y_missing_target, cv=5, scoring='roc_auc').mean()
    
    print(f"📏 Pure Missingness Pattern Model AUC: {cv_auc:.4f}")
    if cv_auc < 0.60:
        print("   💡 点评: 缺失值模式模型的 AUC 极低（接近瞎猜），强力证明了 STARK 的预测能力源于真实的病理学数值，而非“医生爱开检查单”的行为偏倚。")
    else:
        print("   💡 点评: AUC 偏高，说明缺失模式中可能包含预后信号。")
except Exception as e:
    print(f"⚠️ 无法执行缺失模式测试: {e}")

# ==========================================
# 🎯 分析 4：临床效用带宽仿真与 DCA 决策曲线
# ==========================================
print("\n▶️ ANALYSIS 4: Clinical Utility Simulation & Decision Curve Analysis (DCA)")

thresholds = np.linspace(0.01, 0.50, 50) 
N = len(y_true)
prevalence = np.mean(y_true) 

net_benefits_model = []
net_benefits_treat_all = []
net_benefits_treat_none = np.zeros(len(thresholds))

for pt in thresholds:
    preds = (stark_probs >= pt).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    nb_model = (tp / N) - (fp / N) * (pt / (1 - pt))
    net_benefits_model.append(nb_model)
    
    nb_all = prevalence - (1 - prevalence) * (pt / (1 - pt))
    net_benefits_treat_all.append(nb_all)

# 1. 绘制标准的 DCA 曲线 
fig, ax = plt.subplots(figsize=(9, 7), dpi=300)

ax.plot(thresholds, net_benefits_model, color='#d62728', linewidth=2.5, label='STARK Framework')
ax.plot(thresholds, net_benefits_treat_all, color='gray', linestyle='-', linewidth=1.5, label='Treat All')
ax.plot(thresholds, net_benefits_treat_none, color='black', linestyle='--', linewidth=1.5, label='Treat None')

ax.set_xlim([0.01, 0.40]) 
y_min = -0.05
y_max = max(net_benefits_model) + 0.05
ax.set_ylim([y_min, y_max])

ax.set_xlabel('Threshold Probability', fontsize=12, fontweight='bold')
ax.set_ylabel('Net Benefit', fontsize=12, fontweight='bold')
ax.set_title('Decision Curve Analysis for Clinical Utility', fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, linestyle=':', alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig_dca_path = os.path.join(OUTPUT_DIR, "Figure_8_DCA_Curve.png")
plt.savefig(fig_dca_path, dpi=300, bbox_inches='tight')
print(f"✅ Figure 8 (DCA决策曲线图) 已生成。")

# 2. 导出带宽仿真表 Table 8
sim_thresholds = [0.05, 0.10, 0.15, 0.20, 0.25]
utility_results = []
for pt in sim_thresholds:
    preds = (stark_probs >= pt).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    nb = (tp / N) - (fp / N) * (pt / (1 - pt))
    utility_results.append({
        'Action Threshold': f"{pt*100:.0f}%",
        'Net Benefit': f"{nb:.4f}",
        'KPB Triggers per 100 Patients': f"{((tp + fp) / N) * 100:.1f}",
        'True Interceptions (TP)': tp,
        'False Alarms (FP)': fp
    })

excel_utility_path = os.path.join(OUTPUT_DIR, "Table_8_Clinical_Utility.xlsx")
pd.DataFrame(utility_results).to_excel(excel_utility_path, index=False)
print(f"✅ Table 8 (临床效用仿真表) 已生成。")

# ==========================================
# 🎯 分析 5：算法错题集审计 (Error Audit / Failure Mode)
# ==========================================
print("\n▶️ ANALYSIS 5: Error Audit / Edge Cases Extraction")

fn_mask = (y_true == 1) & (stark_probs < CLINICAL_THRESHOLD)
df_fn = df_test[fn_mask].copy()

fp_mask = (y_true == 0) & (stark_probs > 0.80)
df_fp = df_test[fp_mask].copy()

print(f"🔍 提取到极端漏诊病例 (False Negatives < 15%): {len(df_fn)} 例")
print(f"🔍 提取到极端误诊病例 (False Positives > 80%): {len(df_fp)} 例")

excel_error_path = os.path.join(OUTPUT_DIR, "Error_Audit_Cases.xlsx")
with pd.ExcelWriter(excel_error_path) as writer:
    df_fn.to_excel(writer, sheet_name="False_Negatives(Missed)", index=False)
    df_fp.to_excel(writer, sheet_name="False_Positives(FalseAlarms)", index=False)

print(f"✅ 错题集已导出至 Error_Audit_Cases.xlsx，请翻阅这些病人的基线写进 Discussion。")

print("\n" + "="*85)
print(f"🎉 所有的跑分、建表与画图均已完美竣工！")
print(f"📂 请打开服务器上的 {OUTPUT_DIR} 文件夹，收取你的顶刊图表。")
print("="*85)

In [2]:
# -*- coding: utf-8 -*-
"""
Ultimate Clinical & Fairness Audit Package for npj Digital Medicine (V3 Fixed)
- Removed toxic index merging. Uses robust native one-hot encoded features.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import warnings

warnings.filterwarnings('ignore')

print("="*85)
print("🛡️ RUNNING ULTIMATE CLINICAL & FAIRNESS AUDIT PACKAGE (npj V3 Edition) 🛡️")
print("="*85)

# ==========================================
# 0. 基础配置、数据加载与输出文件夹准备
# ==========================================
BASE_DIR = os.getcwd()
DATA_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'test_imputed_random_forest.csv')
ORIGINAL_DATA_FILE = os.path.join(BASE_DIR, 'clean_clinical_data.csv')
STACKING_MODEL_FILE = os.path.join(BASE_DIR, 'Final_Internal_Validation_Optimized', 'model_stacking.pkl')

TARGET_COL = 'PostopAKI'
CLINICAL_THRESHOLD = 0.15 # 论文推荐的床旁预警红线

OUTPUT_DIR = os.path.join(BASE_DIR, "npj_Rebuttal_Outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 图表将保存在独立文件夹中: {OUTPUT_DIR}\n")

try:
    df_test = pd.read_csv(DATA_FILE)
    df_orig = pd.read_csv(ORIGINAL_DATA_FILE) 
    
    # 🌟 核心修复：彻底删除导致空值的 merge 逻辑！
    # 我们直接使用 df_test 已经插补好的高维数据，确保万无一失
    
    y_true = df_test[TARGET_COL].astype(int).values
    model_stacking, expected_cols = joblib.load(STACKING_MODEL_FILE)
    X_test_stark = df_test[expected_cols].fillna(0)
    
    stark_probs = model_stacking.predict_proba(X_test_stark)[:, 1]
    df_test['STARK_Prob'] = stark_probs
    print("✅ 数据与 STARK 模型加载成功，预测完成。")
except Exception as e:
    print(f"❌ 加载失败，请检查路径。报错信息: {e}")
    exit()

# ==========================================
# 🎯 分析 1：算法公平性审计 (图表并茂版)
# ==========================================
print("\n▶️ ANALYSIS 1: Algorithmic Fairness Audit (Equalized Odds)")

def calc_fairness_metrics(y, probs, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    return fnr, fpr

# 🌟 核心修复：直接用插补数据的原生列切分，告别 NaN！
fairness_results = []
subgroups = [
    ("Age >= 65", df_test[df_test['Age'] >= 65]),
    ("Age < 65",  df_test[df_test['Age'] < 65]),
]
if 'Gender_1' in df_test.columns and 'Gender_2' in df_test.columns:
    subgroups.extend([
        ("Male",   df_test[df_test['Gender_1'] == 1]),
        ("Female", df_test[df_test['Gender_2'] == 1])
    ])

for group_name, sub_df in subgroups:
    if len(sub_df) == 0: 
        print(f"⚠️ 警告：亚组 {group_name} 人数为 0，请检查特征名。")
        continue
    y_sub = sub_df[TARGET_COL].astype(int).values
    probs_sub = sub_df['STARK_Prob'].values
    fnr, fpr = calc_fairness_metrics(y_sub, probs_sub, CLINICAL_THRESHOLD)
    fairness_results.append({
        'Subgroup': group_name, 'N': len(sub_df), 
        'FNR': fnr, 'FPR': fpr 
    })

df_fairness = pd.DataFrame(fairness_results)

df_fairness_out = df_fairness.copy()
df_fairness_out['FNR (Miss Rate)'] = df_fairness_out['FNR'].apply(lambda x: f"{x:.3f}")
df_fairness_out['FPR (False Alarm)'] = df_fairness_out['FPR'].apply(lambda x: f"{x:.3f}")
excel_fairness_path = os.path.join(OUTPUT_DIR, "Table_S16_Fairness_Audit.xlsx")
df_fairness_out.drop(columns=['FNR', 'FPR']).to_excel(excel_fairness_path, index=False)
print(f"✅ Table S16 (公平性表格) 已成功保存。")

fig, ax = plt.subplots(figsize=(9, 6), dpi=300)
x = np.arange(len(df_fairness))
width = 0.35

rects1 = ax.bar(x - width/2, df_fairness['FNR'], width, label='False Negative Rate (Miss)', color='#1f77b4', alpha=0.85)
rects2 = ax.bar(x + width/2, df_fairness['FPR'], width, label='False Positive Rate (Alarm)', color='#ff7f0e', alpha=0.85)

ax.set_ylabel('Error Rate', fontsize=12, fontweight='bold')
ax.set_title(f'Algorithmic Fairness Across Demographics (Threshold = {CLINICAL_THRESHOLD*100}%)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(df_fairness['Subgroup'], fontsize=11)
ax.legend(loc='upper left')
ax.grid(axis='y', linestyle=':', alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 在柱子上标注具体数值，防瞎眼
for rect in rects1 + rects2:
    height = rect.get_height()
    ax.annotate(f'{height:.2f}',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3), 
                textcoords="offset points",
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
fig_fairness_path = os.path.join(OUTPUT_DIR, "Figure_S7_Fairness_Audit.png")
plt.savefig(fig_fairness_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Figure S7 (公平性视觉图) 已成功生成并涂色完毕！")

# ==========================================
# 🎯 分析 2：全局特征置换重要性 (Permutation Importance - AUC 修复版)
# ==========================================
print("\n▶️ ANALYSIS 2: Global Permutation Importance (AUC Drop)")

# 🌟 核心修复：使用绝对不受阈值影响的 AUC 代替 F1-score
base_auc = roc_auc_score(y_true, stark_probs)

importances = []
print("   (Calculating permutations... this takes a few seconds)")
top_features = expected_cols[:30] if len(expected_cols) > 30 else expected_cols

for col in top_features:
    X_permuted = X_test_stark.copy()
    # 彻底打乱该列的数据 (模拟“蒙住模型的眼睛”)
    X_permuted[col] = np.random.permutation(X_permuted[col].values)
    
    # 重新预测概率
    perm_probs = model_stacking.predict_proba(X_permuted)[:, 1]
    
    # 计算打乱后的 AUC
    perm_auc = roc_auc_score(y_true, perm_probs)
    
    # 记录 AUC 的下跌幅度 (Drop)
    drop_val = base_auc - perm_auc
    importances.append({'Feature': col, 'Importance (AUC Drop)': drop_val})

# 提取下跌最猛的 Top 15 特征
df_imp = pd.DataFrame(importances).sort_values(by='Importance (AUC Drop)', ascending=True).tail(15) 

# 🎨 开始绘制高颜值的条形图
fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
bars = ax.barh(df_imp['Feature'], df_imp['Importance (AUC Drop)'], color='#d62728', alpha=0.85)

ax.set_xlabel('Decrease in AUC (Permutation Importance)', fontsize=12, fontweight='bold')
ax.set_title('Global Feature Importance via Target Permutation', fontsize=14, fontweight='bold', pad=15)
ax.grid(axis='x', linestyle=':', alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 🌟 在每根柱子旁边标注精确的 AUC 跌幅数值，让审稿人一目了然
for bar in bars:
    width = bar.get_width()
    # 如果特征极度不重要，轻微的随机扰动可能导致负值，做个简单的位置适配
    label_x_pos = width + 0.001 if width >= 0 else width - 0.001
    ha = 'left' if width >= 0 else 'right'
    
    ax.text(label_x_pos, bar.get_y() + bar.get_height()/2, f'{width:.4f}',
            va='center', ha=ha, fontsize=10)

plt.tight_layout()
fig_imp_path = os.path.join(OUTPUT_DIR, "Figure_S6_Permutation_Importance.png")
plt.savefig(fig_imp_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Figure S6 (置换重要性条形图) 已完美修复并生成！")

# ==========================================
# 🎯 分析 3：缺失值模式假信号排除测试
# ==========================================
print("\n▶️ ANALYSIS 3: Missingness Pattern Fake Signal Test")
try:
    orig_features = [c for c in expected_cols if c in df_orig.columns]
    X_missing = df_orig[orig_features].isnull().astype(int)
    
    mapping_dict = {'Yes': 1, 'No': 0, '1': 1, '0': 0, 1: 1, 0: 0, 'yes': 1, 'no': 0}
    y_missing_target = df_orig[TARGET_COL].map(mapping_dict).fillna(0).astype(int).values
    
    lr_missing = LogisticRegression(class_weight='balanced', max_iter=200)
    cv_auc = cross_val_score(lr_missing, X_missing, y_missing_target, cv=5, scoring='roc_auc').mean()
    print(f"📏 Pure Missingness Pattern Model AUC: {cv_auc:.4f}")
except Exception as e:
    print(f"⚠️ 执行缺失模式测试时发生意外错误: {e}")

# ==========================================
# 🎯 分析 4：临床效用复合图 (Panel A: DCA 曲线 | Panel B: 效用热图矩阵)
# ==========================================
print("\n▶️ ANALYSIS 4: Composite Figure for Clinical Utility (DCA + Heatmap Matrix)")

import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

thresholds = np.linspace(0.01, 0.50, 50) 
N = len(y_true)
prevalence = np.mean(y_true) 

net_benefits_model = []
net_benefits_treat_all = []
net_benefits_treat_none = np.zeros(len(thresholds))

# 计算 DCA 的连续数据
for pt in thresholds:
    preds = (stark_probs >= pt).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    nb_model = (tp / N) - (fp / N) * (pt / (1 - pt))
    net_benefits_model.append(nb_model)
    nb_all = prevalence - (1 - prevalence) * (pt / (1 - pt))
    net_benefits_treat_all.append(nb_all)

# 计算热图矩阵的离散截点数据
sim_thresholds = [0.05, 0.10, 0.15, 0.20, 0.25]
hm_nb, hm_tp, hm_fp, hm_triggers = [], [], [], []

for pt in sim_thresholds:
    preds = (stark_probs >= pt).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    nb = (tp / N) - (fp / N) * (pt / (1 - pt))
    
    hm_nb.append(nb)
    hm_tp.append(tp)
    hm_fp.append(fp)
    hm_triggers.append(((tp + fp) / N) * 100)

# 构建原始数值的 DataFrame
df_hm_raw = pd.DataFrame({
    'Net Benefit (Higher is better)': [f"{x:.4f}" for x in hm_nb],
    'True Interceptions (Higher is better)': [f"{int(x)}" for x in hm_tp],
    'False Alarms (Lower is better)': [f"{int(x)}" for x in hm_fp],
    'Triggers per 100 Pts (Lower is better)': [f"{x:.1f}" for x in hm_triggers]
}, index=[f"{int(t*100)}%" for t in sim_thresholds]).T

# 构建用于着色的归一化 DataFrame (映射到 0~1)
# 逻辑：对于获益类（NB, TP），数值越大颜色越深 (归一化为1)
#       对于代价类（FP, Triggers），数值越小颜色越深 (反向归一化)
def min_max_scale(series, invert=False):
    if max(series) == min(series): return [0.5]*len(series)
    scaled = (np.array(series) - min(series)) / (max(series) - min(series))
    return 1 - scaled if invert else scaled

df_hm_color = pd.DataFrame({
    'Net Benefit (Higher is better)': min_max_scale(hm_nb, invert=False),
    'True Interceptions (Higher is better)': min_max_scale(hm_tp, invert=False),
    'False Alarms (Lower is better)': min_max_scale(hm_fp, invert=True),
    'Triggers per 100 Pts (Lower is better)': min_max_scale(hm_triggers, invert=True)
}, index=[f"{int(t*100)}%" for t in sim_thresholds]).T

# 🎨 开始绘制极其华丽的上下拼接复合图 (Panel A + Panel B)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12), dpi=300, gridspec_kw={'height_ratios': [1.5, 1]})

# --- Panel A: DCA 曲线 ---
ax1.plot(thresholds, net_benefits_model, color='#d62728', linewidth=2.5, label='STARK Framework')
ax1.plot(thresholds, net_benefits_treat_all, color='gray', linestyle='-', linewidth=1.5, label='Treat All')
ax1.plot(thresholds, net_benefits_treat_none, color='black', linestyle='--', linewidth=1.5, label='Treat None')

ax1.set_xlim([0.01, 0.40]) 
y_min = -0.05
y_max = max(net_benefits_model) + 0.05
ax1.set_ylim([y_min, y_max])
ax1.set_xlabel('Decision Threshold Probability', fontsize=12, fontweight='bold')
ax1.set_ylabel('Net Benefit', fontsize=12, fontweight='bold')
ax1.set_title('A. Decision Curve Analysis (DCA)', fontsize=14, fontweight='bold', loc='left', pad=15)
ax1.legend(loc='upper right', fontsize=11)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Panel B: 效用热图矩阵 ---
# 定义一个高颜值的顶刊渐变色带 (例如从浅蓝色到深蓝色)
cmap = sns.light_palette("#1f77b4", as_cmap=True)

sns.heatmap(df_hm_color, annot=df_hm_raw, fmt="", cmap=cmap, cbar=False, 
            linewidths=2, ax=ax2, annot_kws={"size": 13, "weight": "bold"})

ax2.set_xlabel('Clinical Intervention Threshold', fontsize=12, fontweight='bold')
ax2.set_title('B. Decision Bandwidth Matrix: Interceptions vs. False Alarms', fontsize=14, fontweight='bold', loc='left', pad=15)

# 微调热图样式
ax2.tick_params(axis='x', labelsize=12)
ax2.tick_params(axis='y', labelsize=11, rotation=0)

plt.tight_layout()
fig_combined_path = os.path.join(OUTPUT_DIR, "Figure_8_Composite_Clinical_Utility.png")
plt.savefig(fig_combined_path, dpi=300, bbox_inches='tight')
plt.close()

# 顺便导出 Excel
df_hm_raw.T.to_excel(os.path.join(OUTPUT_DIR, "Table_8_Clinical_Utility.xlsx"), index=True)

print(f"✅ Figure 8 (DCA曲线 + 热图矩阵 复合图) 已成功生成并保存！")

# ==========================================
# 🎯 分析 5：错题集审计
# ==========================================
print("\n▶️ ANALYSIS 5: Error Audit / Edge Cases Extraction")
fn_mask = (y_true == 1) & (stark_probs < CLINICAL_THRESHOLD)
fp_mask = (y_true == 0) & (stark_probs > 0.80)
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Error_Audit_Cases.xlsx")) as writer:
    df_test[fn_mask].to_excel(writer, sheet_name="False_Negatives_Missed", index=False)
    df_test[fp_mask].to_excel(writer, sheet_name="False_Positives_Alarms", index=False)

print("\n" + "="*85)
print(f"🎉 修复版运行完毕！去看看新生成的 Figure_S7 吧！")
print("="*85)

🛡️ RUNNING ULTIMATE CLINICAL & FAIRNESS AUDIT PACKAGE (npj V3 Edition) 🛡️
📁 图表将保存在独立文件夹中: /home/lei/最新分析AKI/npj_Rebuttal_Outputs

✅ 数据与 STARK 模型加载成功，预测完成。

▶️ ANALYSIS 1: Algorithmic Fairness Audit (Equalized Odds)
✅ Table S16 (公平性表格) 已成功保存。
✅ Figure S7 (公平性视觉图) 已成功生成并涂色完毕！

▶️ ANALYSIS 2: Global Permutation Importance (AUC Drop)
   (Calculating permutations... this takes a few seconds)
✅ Figure S6 (置换重要性条形图) 已完美修复并生成！

▶️ ANALYSIS 3: Missingness Pattern Fake Signal Test
📏 Pure Missingness Pattern Model AUC: 0.5444

▶️ ANALYSIS 4: Clinical Utility Simulation & Decision Curve Analysis (DCA)
✅ Figure 8 (DCA决策曲线图) 已成功生成。

▶️ ANALYSIS 5: Error Audit / Edge Cases Extraction

🎉 修复版运行完毕！去看看新生成的 Figure_S7 吧！


In [4]:
# -*- coding: utf-8 -*-
"""
Nature-Style Standalone Heatmap Matrix for Clinical Utility
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix
import warnings

warnings.filterwarnings('ignore')

# 1. 设置 Nature 风格的全局字体与排版
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

print("="*85)
print("🎨 GENERATING NATURE-STYLE CLINICAL UTILITY HEATMAP 🎨")
print("="*85)

# ==========================================
# 2. 基础配置与数据加载
# ==========================================
BASE_DIR = os.getcwd()
DATA_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'test_imputed_random_forest.csv')
STACKING_MODEL_FILE = os.path.join(BASE_DIR, 'Final_Internal_Validation_Optimized', 'model_stacking.pkl')

TARGET_COL = 'PostopAKI'
OUTPUT_DIR = os.path.join(BASE_DIR, "npj_Rebuttal_Outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    df_test = pd.read_csv(DATA_FILE)
    if 'Unnamed: 0' in df_test.columns: df_test.drop(columns=['Unnamed: 0'], inplace=True)
    if 'ID' in df_test.columns: df_test.drop(columns=['ID'], inplace=True)
    
    y_true = df_test[TARGET_COL].astype(int).values
    model_stacking, expected_cols = joblib.load(STACKING_MODEL_FILE)
    X_test_stark = df_test[expected_cols].fillna(0)
    stark_probs = model_stacking.predict_proba(X_test_stark)[:, 1]
    print("✅ 数据与 STARK 模型加载成功。")
except Exception as e:
    print(f"❌ 加载失败，请检查路径。报错信息: {e}")
    exit()

# ==========================================
# 3. 计算离散阈值下的效用指标
# ==========================================
sim_thresholds = [0.05, 0.10, 0.15, 0.20, 0.25]
N = len(y_true)

hm_nb, hm_tp, hm_fp, hm_triggers = [], [], [], []

for pt in sim_thresholds:
    preds = (stark_probs >= pt).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    nb = (tp / N) - (fp / N) * (pt / (1 - pt))
    
    hm_nb.append(nb)
    hm_tp.append(tp)
    hm_fp.append(fp)
    hm_triggers.append(((tp + fp) / N) * 100)

# 构建展示用的文本 (真实数值)
df_hm_raw = pd.DataFrame({
    'Net Benefit (Higher is better)': [f"{x:.4f}" for x in hm_nb],
    'True Interceptions (Higher is better)': [f"{int(x)}" for x in hm_tp],
    'False Alarms (Lower is better)': [f"{int(x)}" for x in hm_fp],
    'Triggers per 100 Pts (Lower is better)': [f"{x:.1f}" for x in hm_triggers]
}, index=[f"{int(t*100)}%" for t in sim_thresholds]).T

# ==========================================
# 4. 临床意向归一化 (核心算法)
# ==========================================
# 将数值映射到 0~1 区间。值越接近1，临床效果越“好”，颜色越深。
def min_max_scale(series, invert=False):
    if max(series) == min(series): return [0.5]*len(series)
    scaled = (np.array(series) - min(series)) / (max(series) - min(series))
    return 1 - scaled if invert else scaled

df_hm_color = pd.DataFrame({
    'Net Benefit (Higher is better)': min_max_scale(hm_nb, invert=False),
    'True Interceptions (Higher is better)': min_max_scale(hm_tp, invert=False),
    'False Alarms (Lower is better)': min_max_scale(hm_fp, invert=True), # 误诊越低越好 (Invert=True)
    'Triggers per 100 Pts (Lower is better)': min_max_scale(hm_triggers, invert=True) # 干预触发越低越省资源
}, index=[f"{int(t*100)}%" for t in sim_thresholds]).T

# ==========================================
# 5. 绘制 Nature 风格极简热图
# ==========================================
fig, ax = plt.subplots(figsize=(10, 4.5), dpi=300)

# 选用顶刊极简蓝渐变色 (类似 YlGnBu 或 纯净 Blues)
# cmap = sns.color_palette("YlGnBu", as_cmap=True) # 你也可以试试这个配色
cmap = sns.color_palette("Blues", as_cmap=True)

# 绘制热图：利用 linewidths 和 linecolor 创造出 Nature 标志性的“瓷砖感”
sns.heatmap(df_hm_color, 
            annot=df_hm_raw,       # 填入真实数据的文字
            fmt="",                # 取消 seaborn 默认的数值格式化
            cmap=cmap,             # 使用顶刊渐变色
            cbar=False,            # 去掉误导性的统一色条 (因为各行量纲不同)
            linewidths=3,          # 极宽的分割线
            linecolor='white',     # 纯白分割线
            ax=ax, 
            annot_kws={"size": 15, "weight": "bold"}) # 大字号加粗

# 坐标轴美化
ax.set_xlabel('Clinical Intervention Threshold', fontsize=14, fontweight='bold', labelpad=15)
ax.set_title('Decision Bandwidth Matrix: Clinical Utility vs. Resource Allocation', 
             fontsize=16, fontweight='bold', loc='left', pad=20)

# 刻度美化：去除默认的刻度短线，仅保留文字
ax.tick_params(axis='both', which='both', length=0)
ax.tick_params(axis='x', labelsize=13)
ax.tick_params(axis='y', labelsize=12, rotation=0)

# 将图像四周的黑色边框彻底去掉 (Frameless)
for _, spine in ax.spines.items():
    spine.set_visible(False)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "Figure_8_Standalone_Heatmap_Nature.png")
plt.savefig(fig_path, dpi=300, bbox_inches='tight', transparent=False)
plt.close()

print(f"\n✅ 极简顶级 Nature 风格独立热图已生成: {fig_path}")

🎨 GENERATING NATURE-STYLE CLINICAL UTILITY HEATMAP 🎨
✅ 数据与 STARK 模型加载成功。

✅ 极简顶级 Nature 风格独立热图已生成: /home/lei/最新分析AKI/npj_Rebuttal_Outputs/Figure_8_Standalone_Heatmap_Nature.png


In [7]:
# -*- coding: utf-8 -*-
"""
Nature-Style Benchmarking Figures (Separated Panels)
Figure 7a: Grouped Bar Chart for AUC & F1-Score
Figure 7b: Log-Scale Bar Chart for Compute Time (Compute Wall)
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. 设置 Nature 风格的全局排版
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['figure.titlesize'] = 16
plt.rcParams['figure.titleweight'] = 'bold'

print("="*85)
print("🎨 GENERATING NATURE-STYLE BENCHMARKING FIGURES (SEPARATED) 🎨")
print("="*85)

OUTPUT_DIR = os.path.join(os.getcwd(), "npj_Rebuttal_Outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. 模拟数据准备 
# ⚠️ 请务必替换为你真实的测试结果数值！
# ==========================================
models = ['DistilGPT2\n(0.08B)', 'Qwen-2.5\n(1.5B)', 'BioMistral\n(7B)', 'Medical Llama-3\n(8B)', 'STARK\n(Stacking)']

auc_scores = [0.552, 0.685, 0.712, 0.738, 0.865]
f1_scores  = [0.125, 0.320, 0.355, 0.402, 0.510]
time_seconds = [350.5, 1250.0, 5800.0, 7500.0, 0.45] 

x = np.arange(len(models))  
width = 0.35  

# Nature 强调色
colors_auc = ['#B0B0B0', '#909090', '#707070', '#505050', '#D62728'] # 红色突出 STARK
colors_f1  = ['#D0D0D0', '#B8B8B8', '#A0A0A0', '#888888', '#FF7F0E'] # 橙色突出 STARK
colors_time= ['#A0A0A0', '#A0A0A0', '#A0A0A0', '#A0A0A0', '#1F77B4'] # 蓝色突出 STARK

import matplotlib.patches as mpatches

# ==========================================
# 3. 绘制并保存图 7a (判别力对比图)
# ==========================================
print("正在生成 Figure 7a (判别力对比)...")
fig_a, ax_a = plt.subplots(figsize=(8.5, 6), dpi=300)

rects1 = ax_a.bar(x - width/2, auc_scores, width, label='AUC', color=colors_auc, edgecolor='black', linewidth=0.8)
rects2 = ax_a.bar(x + width/2, f1_scores, width, label='F1-Score', color=colors_f1, edgecolor='black', linewidth=0.8)

ax_a.set_ylabel('Performance Metric Score', fontsize=13, fontweight='bold')
ax_a.set_title('A. Head-to-Head Discriminative Performance', fontsize=15, loc='left', pad=15)
ax_a.set_xticks(x)
ax_a.set_xticklabels(models, fontsize=11)
ax_a.set_ylim([0, 1.0])

auc_patch = mpatches.Patch(facecolor='gray', edgecolor='black', label='AUC')
f1_patch = mpatches.Patch(facecolor='lightgray', edgecolor='black', label='F1-Score')
ax_a.legend(handles=[auc_patch, f1_patch], loc='upper left', frameon=False, fontsize=12)

for rects in [rects1, rects2]:
    for rect in rects:
        height = rect.get_height()
        ax_a.annotate(f'{height:.3f}',
                     xy=(rect.get_x() + rect.get_width() / 2, height),
                     xytext=(0, 3), 
                     textcoords="offset points",
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

ax_a.spines['top'].set_visible(False)
ax_a.spines['right'].set_visible(False)
ax_a.yaxis.grid(True, linestyle='--', alpha=0.4) 

plt.tight_layout()
fig_a_path = os.path.join(OUTPUT_DIR, "Figure_7a_Discriminative_Performance.png")
fig_a.savefig(fig_a_path, dpi=300, bbox_inches='tight')
plt.close(fig_a)
print(f"✅ Figure 7a 已保存至: {fig_a_path}")


# ==========================================
# 4. 绘制并保存图 7b (算力开销对数图)
# ==========================================
print("正在生成 Figure 7b (算力开销对数轴)...")
fig_b, ax_b = plt.subplots(figsize=(7, 6), dpi=300)

bars = ax_b.bar(x, time_seconds, width=0.6, color=colors_time, edgecolor='black', linewidth=0.8)

ax_b.set_yscale('log')
ax_b.set_ylabel('Inference Time (Seconds, Log Scale)', fontsize=13, fontweight='bold')
ax_b.set_title('B. Computational Overhead (The "Compute Wall")', fontsize=15, loc='left', pad=15)
ax_b.set_xticks(x)
ax_b.set_xticklabels(models, fontsize=11)

ax_b.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: '{:g}'.format(y)))

for bar in bars:
    height = bar.get_height()
    text_y = height * 1.2 if height < 10 else height * 1.15
    ax_b.text(bar.get_x() + bar.get_width()/2, text_y, 
             f'{height:.2f}s' if height < 10 else f'{int(height)}s', 
             ha='center', va='bottom', fontsize=10, fontweight='bold', color='black')

ax_b.axhline(y=3600, color='red', linestyle=':', linewidth=1.5, alpha=0.7)
ax_b.text(x[-1], 3600*1.3, '1 Hour', color='red', ha='right', va='bottom', fontsize=10, fontweight='bold')

ax_b.spines['top'].set_visible(False)
ax_b.spines['right'].set_visible(False)
ax_b.yaxis.grid(True, which='both', linestyle=':', alpha=0.4) 

plt.tight_layout()
fig_b_path = os.path.join(OUTPUT_DIR, "Figure_7b_Computational_Overhead.png")
fig_b.savefig(fig_b_path, dpi=300, bbox_inches='tight')
plt.close(fig_b)
print(f"✅ Figure 7b 已保存至: {fig_b_path}")

print("\n🎉 两张独立的极简高级图表生成完毕！")

🎨 GENERATING NATURE-STYLE BENCHMARKING FIGURES (SEPARATED) 🎨
正在生成 Figure 7a (判别力对比)...
✅ Figure 7a 已保存至: /home/lei/最新分析AKI/npj_Rebuttal_Outputs/Figure_7a_Discriminative_Performance.png
正在生成 Figure 7b (算力开销对数轴)...
✅ Figure 7b 已保存至: /home/lei/最新分析AKI/npj_Rebuttal_Outputs/Figure_7b_Computational_Overhead.png

🎉 两张独立的极简高级图表生成完毕！
